# Data Exploration & EDA
## California Property Close Price Prediction

This notebook contains exploratory data analysis (EDA) for the CRMLS sold listings dataset. We'll examine distributions, patterns, key fields, and data quality to inform preprocessing and modeling decisions.

**Project:** IDX Exchange — Summer 2026 Data Science Internship  
**Target Variable:** `ClosePrice` (final sale price)  
**Dataset:** Single-family residential properties in California

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load Dataset

In [ ]:
# Define data directory
data_dir = Path('../data/cleaned')

# List available CSV files
if data_dir.exists():
    csv_files = list(data_dir.glob('*.csv'))
    print(f"Found {len(csv_files)} CSV files in {data_dir}:")
    for file in csv_files:
        print(f"  - {file.name}")
else:
    print(f"Data directory not found at {data_dir}")
    print("Please ensure raw data has been cleaned and saved to data/cleaned/")

# Load the dataset (adjust filename based on what's available)
try:
    # Try to load the main dataset
    df = pd.read_csv(list(data_dir.glob('*.csv'))[0]) if csv_files else None
    if df is not None:
        print(f"\nLoaded dataset shape: {df.shape}")
    else:
        print("No dataset loaded yet. Please ensure CSV files are in data/cleaned/")
except Exception as e:
    print(f"Error loading dataset: {e}")
    df = None

## 3. Inspect Dataset Structure

In [ ]:
if df is not None:
    # Dataset shape
    print("=" * 80)
    print("DATASET SHAPE & BASIC INFO")
    print("=" * 80)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
    
    # Data types
    print("DATA TYPES:")
    print(df.dtypes)
    
    # Missing values
    print("\n" + "=" * 80)
    print("MISSING VALUES")
    print("=" * 80)
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'Column': missing.index,
        'Missing_Count': missing.values,
        'Missing_Percentage': missing_pct.values
    }).sort_values('Missing_Count', ascending=False)
    print(missing_df[missing_df['Missing_Count'] > 0].to_string())
    
    # Display first few rows
    print("\n" + "=" * 80)
    print("FIRST 5 ROWS")
    print("=" * 80)
    print(df.head())
    
    # Basic statistics
    print("\n" + "=" * 80)
    print("DESCRIPTIVE STATISTICS")
    print("=" * 80)
    print(df.describe())

## 4. Clean and Preprocess Data

In [ ]:
if df is not None:
    df_clean = df.copy()
    
    print("=" * 80)
    print("DATA CLEANING STEPS")
    print("=" * 80)
    
    # Step 1: Handle missing values
    print("\n1. HANDLING MISSING VALUES")
    print("-" * 40)
    
    # Check for target variable (ClosePrice)
    if 'ClosePrice' in df_clean.columns:
        initial_rows = len(df_clean)
        df_clean = df_clean.dropna(subset=['ClosePrice'])
        print(f"Removed {initial_rows - len(df_clean)} rows with missing ClosePrice")
    
    # Fill numeric columns with median where missing < 5%
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        missing_pct = (df_clean[col].isnull().sum() / len(df_clean)) * 100
        if 0 < missing_pct < 5:
            median_val = df_clean[col].median()
            df_clean[col].fillna(median_val, inplace=True)
            print(f"  Filled {col} with median: {median_val:.2f}")
    
    # Step 2: Check data types
    print("\n2. DATA TYPE CONVERSIONS")
    print("-" * 40)
    print("Numeric columns:", list(numeric_cols))
    
    # Step 3: Remove duplicates
    print("\n3. REMOVING DUPLICATES")
    print("-" * 40)
    initial_rows = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    print(f"Removed {initial_rows - len(df_clean)} duplicate rows")
    
    print(f"\nCleaned dataset shape: {df_clean.shape}")
    print(f"Rows removed: {df.shape[0] - df_clean.shape[0]}")
else:
    print("Dataset not loaded. Skipping cleaning steps.")

## 5. Visualize Key Variables

In [ ]:
if df_clean is not None:
    # Get numeric columns for visualization
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    
    if 'ClosePrice' in numeric_cols:
        # 1. Distribution of target variable
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Histogram
        axes[0].hist(df_clean['ClosePrice'], bins=50, edgecolor='black', alpha=0.7, color='skyblue')
        axes[0].set_xlabel('Close Price ($)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Distribution of Close Price')
        axes[0].grid(True, alpha=0.3)
        
        # Log scale histogram (if helpful for skewed distribution)
        axes[1].hist(np.log10(df_clean['ClosePrice']), bins=50, edgecolor='black', alpha=0.7, color='lightcoral')
        axes[1].set_xlabel('Log10 Close Price')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Distribution of Log(Close Price)')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # 2. Correlation with target
        print("\n" + "=" * 80)
        print("CORRELATION WITH TARGET VARIABLE (ClosePrice)")
        print("=" * 80)
        if len(numeric_cols) > 1:
            correlations = df_clean[numeric_cols].corr()['ClosePrice'].sort_values(ascending=False)
            print(correlations)
            
            # Visualize top correlations
            fig, ax = plt.subplots(figsize=(10, 6))
            top_corr = correlations[1:11]  # Top 10 features (excluding ClosePrice itself)
            top_corr.plot(kind='barh', ax=ax, color='teal')
            ax.set_xlabel('Correlation Coefficient')
            ax.set_title('Top 10 Features Correlated with Close Price')
            plt.tight_layout()
            plt.show()
    
    # 3. Distribution of key numeric features
    if len(numeric_cols) > 0:
        print("\n" + "=" * 80)
        print("DISTRIBUTIONS OF KEY FEATURES")
        print("=" * 80)
        
        # Select up to 4 non-target numeric columns
        features_to_plot = [col for col in numeric_cols if col != 'ClosePrice'][:4]
        
        if features_to_plot:
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            axes = axes.flatten()
            
            for idx, col in enumerate(features_to_plot):
                axes[idx].hist(df_clean[col].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
                axes[idx].set_xlabel(col)
                axes[idx].set_ylabel('Frequency')
                axes[idx].set_title(f'Distribution of {col}')
                axes[idx].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
else:
    print("No cleaned dataset available for visualization.")

## 6. Summarize Insights & Next Steps

In [ ]:
if df_clean is not None:
    print("=" * 80)
    print("EXPLORATION SUMMARY & KEY FINDINGS")
    print("=" * 80)
    
    print("\n### DATASET OVERVIEW ###")
    print(f"- Total Records: {len(df_clean):,}")
    print(f"- Total Features: {df_clean.shape[1]}")
    print(f"- Numeric Features: {len(df_clean.select_dtypes(include=[np.number]).columns)}")
    print(f"- Categorical Features: {len(df_clean.select_dtypes(include=['object']).columns)}")
    
    print("\n### DATA QUALITY ###")
    total_missing = df_clean.isnull().sum().sum()
    total_cells = df_clean.shape[0] * df_clean.shape[1]
    print(f"- Missing Values: {total_missing:,} ({(total_missing/total_cells)*100:.2f}%)")
    print(f"- Duplicate Rows: {len(df) - len(df_clean):,}")
    
    if 'ClosePrice' in df_clean.columns:
        print("\n### TARGET VARIABLE (ClosePrice) ###")
        print(f"- Mean: ${df_clean['ClosePrice'].mean():,.2f}")
        print(f"- Median: ${df_clean['ClosePrice'].median():,.2f}")
        print(f"- Std Dev: ${df_clean['ClosePrice'].std():,.2f}")
        print(f"- Min: ${df_clean['ClosePrice'].min():,.2f}")
        print(f"- Max: ${df_clean['ClosePrice'].max():,.2f}")
    
    print("\n### NEXT STEPS ###")
    print("1. Review feature engineering opportunities (e.g., property age, ratios)")
    print("2. Investigate outliers and extreme values")
    print("3. Analyze temporal patterns (CloseDate trends)")
    print("4. Explore geographic patterns (Latitude/Longitude)")
    print("5. Prepare features for modeling in 02_preprocessing.ipynb")
    
else:
    print("\nNo dataset loaded. Please ensure CSV files are present in data/cleaned/")
    print("\nTO GET STARTED:")
    print("1. Obtain CRMLS sold listings data from IDX Exchange FTP")
    print("2. Filter for PropertyType='Residential', PropertySubType='SingleFamilyResidence'")
    print("3. Save to data/cleaned/ folder")
    print("4. Re-run this notebook")